# ЛР № 3. Профилирование и качество Bike Sharing

Участок `raw → staging`. Одна строка `hour.csv` - один час работы системы проката.
Вопрос из [Project Brief](../docs/project_brief.md): в какие часы суток, дни недели и сезоны
наблюдаются наибольшие и наименьшие значения почасового спроса (`cnt`), и как в этих срезах
различается соотношение `casual` и `registered`.

Бизнес-ключ: `(dteday, hr)`. Отдельного идентификатора поездки в наборе нет - данные уже
агрегированы до часа, поэтому уникальность проверяется по ключу, а не по `instant`.

`day.csv` используется только как справочник для проверки согласованности дневных и почасовых сумм.

## Шаг 1. Входные данные и подключение

ingested_on — логическая дата загрузки, а не период данных; аналитический период - снимок 2011-01-01…2012-12-31; выходной префикс должен соответствовать ingested_on.

In [1]:
import os
import sys
from html import escape
from pathlib import Path

import boto3
import duckdb
from dotenv import load_dotenv
from IPython.display import HTML, display

PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "pyproject.toml").is_file() and (parent / "src/analysis/read_raw.py").is_file()
)
DATE_INGESTED_ON = "2026-11-02"
load_dotenv(PROJECT_ROOT / ".env", override=True)
required = ("S3_ENDPOINT", "S3_ACCESS_KEY", "S3_SECRET_KEY")
missing = [name for name in required if not os.environ.get(name)]
if missing:
    raise ValueError("Не заданы настройки: " + ", ".join(missing))

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from analysis.read_raw import configure_s3

raw_path = f"s3://raw/bike_sharing/ingested_on={DATE_INGESTED_ON}/hour.csv"
day_path = f"s3://raw/bike_sharing/ingested_on={DATE_INGESTED_ON}/day.csv"
period_start = "2011-01-01"
period_end = "2013-01-01"
output_prefix = f"s3://staging/bike_sharing/source_ingested_on={DATE_INGESTED_ON}"


def literal(value):
    return "'" + value.replace("'", "''") + "'"


def show_table(headers, rows):
    header = "".join(f"<th>{escape(str(item))}</th>" for item in headers)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(item))}</td>" for item in row) + "</tr>" for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))


connection = duckdb.connect()
configure_s3(connection)

connection.execute(f"""
    CREATE OR REPLACE TEMP VIEW raw_hour AS
    SELECT * FROM read_csv_auto({literal(raw_path)}, header = true, hive_partitioning = false)
""")
connection.execute(f"""
    CREATE OR REPLACE TEMP VIEW raw_day AS
    SELECT * FROM read_csv_auto({literal(day_path)}, header = true, hive_partitioning = false)
""")

show_table(
    ["Входной файл", "Период", "Выход"],
    [(raw_path, f"{period_start}..{period_end}", output_prefix)],
)
show_table(
    ["dteday", "hr", "cnt", "casual", "registered"],
    connection.execute("""
    SELECT dteday, hr, cnt, casual, registered FROM raw_hour LIMIT 5
""").fetchall(),
)

Входной файл,Период,Выход
s3://raw/bike_sharing/ingested_on=2026-11-02/hour.csv,2011-01-01..2013-01-01,s3://staging/bike_sharing/source_ingested_on=2026-11-02


dteday,hr,cnt,casual,registered
2011-01-01,0,16,3,13
2011-01-01,1,40,8,32
2011-01-01,2,32,5,27
2011-01-01,3,13,3,10
2011-01-01,4,1,0,1


## Шаг 2. Профиль исходных данных

Выбраны поля dteday, hr, cnt, casual, registered, weathersit, season, weekday - временной ключ, основная метрика и доли пользователей, категориальный срез по погоде, сезон и день недели. Профиль считается по полям, используемым в правилах.

In [2]:
raw_count = connection.execute("SELECT COUNT(*) FROM raw_hour").fetchone()[0]
schema = {row[0]: row[1] for row in connection.execute("DESCRIBE raw_hour").fetchall()}
show_table(["Строк", "Полей"], [(raw_count, len(schema))])

selected_fields = ("dteday", "hr", "cnt", "casual", "registered", "weathersit", "season", "weekday")
profile_rows = []
for field in selected_fields:
    missing_count = connection.execute(
        f"SELECT COUNT(*) FROM raw_hour WHERE {field} IS NULL"
    ).fetchone()[0]
    examples = connection.execute(
        f"SELECT DISTINCT {field} FROM raw_hour WHERE {field} IS NOT NULL ORDER BY 1 LIMIT 2"
    ).fetchall()
    profile_rows.append(
        (
            field,
            schema[field],
            ", ".join(str(row[0]) for row in examples),
            raw_count,
            missing_count,
            f"{missing_count / raw_count:.3%}" if raw_count else "N/A",
        )
    )
show_table(["Поле", "Тип", "Примеры", "Проверено", "Пропусков", "Доля"], profile_rows)

Строк,Полей
17379,17


Поле,Тип,Примеры,Проверено,Пропусков,Доля
dteday,DATE,"2011-01-01, 2011-01-02",17379,0,0.000%
hr,BIGINT,"0, 1",17379,0,0.000%
cnt,BIGINT,"1, 2",17379,0,0.000%
casual,BIGINT,"0, 1",17379,0,0.000%
registered,BIGINT,"0, 1",17379,0,0.000%
weathersit,BIGINT,"1, 2",17379,0,0.000%
season,BIGINT,"1, 2",17379,0,0.000%
weekday,BIGINT,"0, 1",17379,0,0.000%


In [3]:
key_rows = connection.execute("""
    SELECT COUNT(*) FROM (
        SELECT dteday, hr FROM raw_hour GROUP BY 1, 2 HAVING COUNT(*) > 1
    )
""").fetchone()[0]
excess_rows = connection.execute("""
    SELECT COALESCE(sum(c - 1), 0) FROM (
        SELECT dteday, hr, COUNT(*) AS c FROM raw_hour GROUP BY 1, 2 HAVING COUNT(*) > 1
    )
""").fetchone()[0]
exact_duplicates = connection.execute("""
    SELECT COUNT(*) - (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM raw_hour)) FROM raw_hour
""").fetchone()[0]
coverage = connection.execute("""
    SELECT COUNT(DISTINCT dteday) AS days, COUNT(*) AS rows,
           COUNT(DISTINCT dteday) * 24 - COUNT(*) AS missing_hours,
           min(dteday), max(dteday)
    FROM raw_hour
""").fetchone()
show_table(
    [
        "Ключей с повторами",
        "Избыточных строк",
        "Полных дублей",
        "Дней",
        "Строк",
        "Недостаёт часов",
        "min dteday",
        "max dteday",
    ],
    [(key_rows, excess_rows, exact_duplicates, *coverage)],
)

hours = connection.execute("""
    SELECT hr, COUNT(*) AS rows, sum(cnt) AS rides, round(avg(cnt), 1) AS avg_cnt
    FROM raw_hour GROUP BY 1 ORDER BY 1
""").fetchall()
show_table(["Час", "Записей", "Прокатов", "Средний cnt"], hours)

Ключей с повторами,Избыточных строк,Полных дублей,Дней,Строк,Недостаёт часов,min dteday,max dteday
0,0,0,731,17379,165,2011-01-01,2012-12-31


Час,Записей,Прокатов,Средний cnt
0,726,39130,53.9
1,724,24164,33.4
2,715,16352,22.9
3,697,8174,11.7
4,697,4428,6.4
5,717,14261,19.9
6,725,55132,76.0
7,727,154171,212.1
8,727,261001,359.0
9,727,159438,219.3


### Наблюдения для отчёта

- Покрытие неполное: 731 день × 24 часа = 17 544 возможных слотов, фактически 17 379 строк — отсутствуют 165 часовых записей. Это ограничивает средний спрос на час и сверку с `day.csv`.
- Ключ `(dteday, hr)` уникален (0 повторов, 0 полных дублей), пропусков в выбранных полях нет. Спрос концентрируется в часах пик: 17:00 — 336 860 прокатов, 18:00 — 309 772, при минимуме в 04:00 — 4 428 прокатов (средний `cnt` 6,4). Эти срезы — основа ответа на основной вопрос Brief.

## Шаг 3. Правила качества

Семь правил ниже: уникальность ключа, полнота обязательных полей, допустимость `hr`, допустимость категорий
(`weathersit`/`season`/`weekday`), согласованность `cnt = casual + registered`, неотрицательность счётчиков, согласованность часов `sum(day.csv) == sum(hour.csv)`.

Для каждого правила: id, поля, проверяемые строки, условие, порог (переменная в ячейке), критичность, действие
и связь с задачей. Порог оценивает набор; действие применяется к каждой нарушившей строке независимо от
превышения порога. Доля считается внутри проверяемой совокупности.

In [4]:
rule_results = {}


def show_rule(rule_id, checked, violations, maximum, action, unit="строк"):
    share = f"{violations / checked:.3%}" if checked else "N/A"
    comparison = ("Превышен" if violations > maximum else "Не превышен") if checked else "N/A"
    row = (
        rule_id,
        f"{checked} {unit}",
        violations,
        share,
        f"count ≤ {maximum}",
        comparison,
        action,
    )
    rule_results[rule_id] = row
    show_table(
        ["Правило", "Проверено", "Нарушений", "Доля", "Порог", "Сравнение", "Действие"], [row]
    )

### Правило 1. `hour_key_unique` - уникальность ключа `(dteday, hr)`

- Поля: `(dteday, hr)` - бизнес-ключ строки.
- Проверяемые строки: все записи `raw_hour`.
- Условие: на каждую пару `(dteday, hr)` приходится не более одной строки.
- Порог: число нарушений; переменная `max_duplicate_keys`.
- Критичность: `critical`.
- Действие: `quarantine`.
- Связь с задачей: повтор часа завышает `sum(cnt)` и `avg(cnt)` в разрезе часа - основной метрики вопроса.

In [5]:
max_duplicate_keys = 0
checked, violations = connection.execute("""
    SELECT (SELECT count(*) FROM raw_hour),
           (SELECT count(*) FROM (
               SELECT dteday, hr FROM raw_hour GROUP BY 1, 2 HAVING count(*) > 1
           ))
""").fetchone()
show_rule("hour_key_unique", checked, violations, max_duplicate_keys, "quarantine")

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
hour_key_unique,17379 строк,0,0.000%,count ≤ 0,Не превышен,quarantine


### Правило 2. `required_fields_present` - полнота обязательных полей

- Поля: `dteday`, `hr`, `cnt`, `casual`, `registered`.
- Проверяемые строки: все записи `raw_hour`.
- Условие: все пять полей заполнены.
- Порог: число нарушений; переменная `max_missing_required`.
- Критичность: `critical`.
- Действие: `quarantine`.
- Связь с задачей: без них не считается ни спрос, ни доли пользователей, ни часовой срез.

In [6]:
max_missing_required = 0
checked, violations = connection.execute("""
    SELECT count(*), count(*) FILTER (
        WHERE dteday IS NULL OR hr IS NULL OR cnt IS NULL
           OR casual IS NULL OR registered IS NULL
    ) FROM raw_hour
""").fetchone()
show_rule("required_fields_present", checked, violations, max_missing_required, "quarantine")

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
required_fields_present,17379 строк,0,0.000%,count ≤ 0,Не превышен,quarantine


### Правило 3. `hr_in_range` - допустимость значения `hr`

- Поле: `hr`.
- Проверяемые строки: записи с заполненным `hr`.
- Условие: `0 ≤ hr ≤ 23`.
- Порог: число нарушений; переменная `max_bad_hr`.
- Критичность: `critical`.
- Действие: `quarantine`.
- Связь с задачей: `hr` - ключевой временной срез основного вопроса; выход за диапазон делает срез неверным.

In [7]:
max_bad_hr = 0
checked, violations = connection.execute("""
    SELECT count(*), count(*) FILTER (WHERE hr < 0 OR hr > 23)
    FROM raw_hour WHERE hr IS NOT NULL
""").fetchone()
show_rule("hr_in_range", checked, violations, max_bad_hr, "quarantine")

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
hr_in_range,17379 строк,0,0.000%,count ≤ 0,Не превышен,quarantine


### Правило 4. `weathersit_in_domain` - допустимость категориальных кодов

- Поля: `weathersit`, `season`, `weekday`.
- Проверяемые строки: все записи `raw_hour`.
- Условие: `weathersit ∈ {1, 2, 3, 4}`, `season ∈ {1, 2, 3, 4}`, `weekday ∈ {0..6}`.
- Порог: число нарушений; переменная `max_bad_category`.
- Критичность: `warning`.
- Действие: `flag`.
- Связь с задачей: срезы по погоде и сезону из уточняющего вопроса 1. Запись остаётся в `staging` с явным признаком нарушения - такие строки легко исключить на этапе анализа.

In [8]:
max_bad_category = 0
checked, violations = connection.execute("""
    SELECT count(*), count(*) FILTER (
        WHERE weathersit NOT IN (1, 2, 3, 4)
           OR season NOT IN (1, 2, 3, 4)
           OR weekday NOT IN (0, 1, 2, 3, 4, 5, 6)
           OR weathersit IS NULL OR season IS NULL OR weekday IS NULL
    ) FROM raw_hour
""").fetchone()
show_rule("weathersit_in_domain", checked, violations, max_bad_category, "flag")

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
weathersit_in_domain,17379 строк,0,0.000%,count ≤ 0,Не превышен,flag


### Правило 5. `cnt_equals_casual_plus_registered` - согласованность полей

- Поля: связь `cnt`, `casual`, `registered`.
- Проверяемые строки: все три поля заполнены.
- Условие: `cnt = casual + registered`.
- Порог: число нарушений; переменная `max_inconsistent_cnt`.
- Критичность: `critical`.
- Действие: `quarantine`.
- Связь с задачей: основная метрика и доли `casual`/`registered` должны быть непротиворечивы - это гипотеза качества из `Project Brief`.

In [9]:
max_inconsistent_cnt = 0
checked, violations = connection.execute("""
    SELECT count(*), count(*) FILTER (WHERE cnt <> casual + registered)
    FROM raw_hour
    WHERE cnt IS NOT NULL AND casual IS NOT NULL AND registered IS NOT NULL
""").fetchone()
show_rule(
    "cnt_equals_casual_plus_registered", checked, violations, max_inconsistent_cnt, "quarantine"
)

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
cnt_equals_casual_plus_registered,17379 строк,0,0.000%,count ≤ 0,Не превышен,quarantine


### Правило 6. `non_negative_counts` - допустимость диапазона счётчиков

- Поля: `cnt`, `casual`, `registered`.
- Проверяемые строки: записи, где хотя бы одно из полей заполнено.
- Условие: все заполненные значения ≥ 0.
- Порог: число нарушений; переменная `max_negative_counts`.
- Критичность: `critical`.
- Действие: `quarantine`.
- Связь с задачей: отрицательный счётчик ломает `sum`, `avg` и доли пользователей.

In [10]:
max_negative_counts = 0
checked, violations = connection.execute("""
    SELECT count(*), count(*) FILTER (
        WHERE cnt < 0 OR casual < 0 OR registered < 0
    ) FROM raw_hour
    WHERE cnt IS NOT NULL OR casual IS NOT NULL OR registered IS NOT NULL
""").fetchone()
show_rule("non_negative_counts", checked, violations, max_negative_counts, "quarantine")

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
non_negative_counts,17379 строк,0,0.000%,count ≤ 0,Не превышен,quarantine


### Правило 7. hour_day_cnt_consistent - согласованность day.csv с hour.csv
- Поля: sum(cnt) по dteday в hour.csv vs cnt в day.csv.
- Проверяемые строки: 731 день.
- Условие: совпадения суммы часов в day.csv и в hour.csv.
- Порог: число нарушений; переменная `max_inconsistent_days`.
- Критичность: `info`.
- Действие: `report_only`.
- Связь с задачей: связь с уточняющим вопросом 2 из Project Brief. report_only, потому что это характеристика снимка, а не «плохая» часовая запись.

In [11]:
max_inconsistent_days = 0
checked, violations = connection.execute("""
    WITH hourly AS (
        SELECT dteday, sum(cnt) AS hour_cnt FROM raw_hour GROUP BY 1
    ), daily AS (
        SELECT dteday, cnt AS day_cnt FROM raw_day
    )
    SELECT count(*),
           count(*) FILTER (WHERE hourly.hour_cnt IS DISTINCT FROM daily.day_cnt)
    FROM daily LEFT JOIN hourly USING (dteday)
""").fetchone()
show_rule(
    "hour_day_cnt_consistent",
    checked,
    violations,
    max_inconsistent_days,
    "report_only",
    unit="дней",
)

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
hour_day_cnt_consistent,731 дней,0,0.000%,count ≤ 0,Не превышен,report_only


## Шаг 4. Staging и Карантин
`staging` — без строк с критическими нарушениями; `flag`-правило даёт Boolean-признак; карантин сохраняет исходные поля, `quality_rule_ids` (через ;) и `quarantine_reason`; при нескольких причинах запись попадает в карантин один раз; карантин приоритетнее флага.

In [12]:
connection.execute("""
    CREATE OR REPLACE TEMP VIEW checked_data AS
    WITH keyed AS (
        SELECT *, count(*) OVER (PARTITION BY dteday, hr) AS key_rows FROM raw_hour
    )
    SELECT *,
        key_rows > 1 AS duplicate_key,
        (dteday IS NULL OR hr IS NULL OR cnt IS NULL
         OR casual IS NULL OR registered IS NULL) AS missing_required,
        COALESCE(hr < 0 OR hr > 23, false) AS bad_hr,
        COALESCE(cnt <> casual + registered, false) AS inconsistent_cnt,
        COALESCE(cnt < 0 OR casual < 0 OR registered < 0, false) AS negative_counts,
        COALESCE(weathersit NOT IN (1, 2, 3, 4) OR weathersit IS NULL
                 OR season NOT IN (1, 2, 3, 4) OR season IS NULL
                 OR weekday NOT IN (0, 1, 2, 3, 4, 5, 6) OR weekday IS NULL, false) AS bad_category
    FROM keyed
""");

In [13]:
connection.execute("""
    CREATE OR REPLACE TEMP VIEW staging_data AS
    SELECT * EXCLUDE (key_rows, duplicate_key, missing_required, bad_hr,
                      inconsistent_cnt, negative_counts, bad_category),
           bad_category AS quality_weathersit_in_domain_violated
    FROM checked_data
    WHERE NOT (duplicate_key OR missing_required OR bad_hr OR inconsistent_cnt OR negative_counts)
""");

In [14]:
connection.execute("""
    CREATE OR REPLACE TEMP VIEW quarantine_data AS
    SELECT * EXCLUDE (key_rows, duplicate_key, missing_required, bad_hr,
                      inconsistent_cnt, negative_counts, bad_category),
        concat(
            CASE WHEN duplicate_key THEN 'hour_key_unique;' ELSE '' END,
            CASE WHEN missing_required THEN 'required_fields_present;' ELSE '' END,
            CASE WHEN bad_hr THEN 'hr_in_range;' ELSE '' END,
            CASE WHEN inconsistent_cnt THEN 'cnt_equals_casual_plus_registered;' ELSE '' END,
            CASE WHEN negative_counts THEN 'non_negative_counts;' ELSE '' END
        ) AS quality_rule_ids,
        concat(
            CASE WHEN duplicate_key THEN 'Повтор ключа (dteday, hr); ' ELSE '' END,
            CASE WHEN missing_required THEN 'Не заполнены обязательные поля; ' ELSE '' END,
            CASE WHEN bad_hr THEN 'Час вне диапазона 0-23; ' ELSE '' END,
            CASE WHEN inconsistent_cnt THEN 'cnt не равен casual + registered; ' ELSE '' END,
            CASE WHEN negative_counts THEN 'Отрицательный счётчик прокатов; ' ELSE '' END
        ) AS quarantine_reason
    FROM checked_data
    WHERE duplicate_key OR missing_required OR bad_hr OR inconsistent_cnt OR negative_counts
""")
show_table(
    ["Набор", "Строк"],
    [
        (name, connection.execute(f"SELECT count(*) FROM {name}").fetchone()[0])
        for name in ("raw_hour", "staging_data", "quarantine_data")
    ],
)

Набор,Строк
raw_hour,17379
staging_data,17379
quarantine_data,0


Карантин пуст и все строки переходят в staging

In [15]:
show_table(
    ["Правило", "Проверено", "Нарушений", "Доля", "Порог", "Сравнение", "Действие"],
    rule_results.values(),
)

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
hour_key_unique,17379 строк,0,0.000%,count ≤ 0,Не превышен,quarantine
required_fields_present,17379 строк,0,0.000%,count ≤ 0,Не превышен,quarantine
hr_in_range,17379 строк,0,0.000%,count ≤ 0,Не превышен,quarantine
weathersit_in_domain,17379 строк,0,0.000%,count ≤ 0,Не превышен,flag
cnt_equals_casual_plus_registered,17379 строк,0,0.000%,count ≤ 0,Не превышен,quarantine
non_negative_counts,17379 строк,0,0.000%,count ≤ 0,Не превышен,quarantine
hour_day_cnt_consistent,731 дней,0,0.000%,count ≤ 0,Не превышен,report_only


### Запись двух результатов в S3

Следующая ячейка заменяет только `staging.parquet` и `quarantine.parquet`.

In [16]:
source_date = raw_path.split("/ingested_on=", 1)[1].split("/", 1)[0]
expected_prefix = f"s3://staging/bike_sharing/source_ingested_on={source_date}"
if output_prefix != expected_prefix:
    raise ValueError("Выходной префикс не соответствует исходному снимку.")

endpoint = os.environ["S3_ENDPOINT"]
client = boto3.client(
    "s3",
    endpoint_url=endpoint if endpoint.startswith("http") else f"http://{endpoint}",
    aws_access_key_id=os.environ["S3_ACCESS_KEY"],
    aws_secret_access_key=os.environ["S3_SECRET_KEY"],
    region_name="us-east-1",
)
prefix = output_prefix.removeprefix("s3://staging/")
for name in ("staging", "quarantine"):
    key = f"{prefix}/{name}.parquet"
    client.delete_object(Bucket="staging", Key=key)
    connection.execute(
        f"COPY (SELECT * FROM {name}_data) TO {literal('s3://staging/' + key)} (FORMAT PARQUET)"
    )
show_table(
    ["Результат", "S3-путь"],
    [(n, f"{output_prefix}/{n}.parquet") for n in ("staging", "quarantine")],
)

Результат,S3-путь
staging,s3://staging/bike_sharing/source_ingested_on=2026-11-02/staging.parquet
quarantine,s3://staging/bike_sharing/source_ingested_on=2026-11-02/quarantine.parquet


<a id="verify"></a>
## Шаг 6. Проверка сохранённых файлов
Читаем оба файла непосредственно из S3.
Проверяем баланс, отсутствие критических нарушений, правильность флага и причины карантина.
Количество полей при чтении может включать `source_ingested_on` из выходного пути -
его нет внутри Parquet-файлов.

In [17]:
for name in ("staging", "quarantine"):
    connection.execute(f"""
        CREATE OR REPLACE TEMP VIEW saved_{name} AS
        SELECT * FROM read_parquet({literal(output_prefix + '/' + name + '.parquet')})
    """)

staging_count = connection.execute("SELECT count(*) FROM saved_staging").fetchone()[0]
quarantine_count = connection.execute("SELECT count(*) FROM saved_quarantine").fetchone()[0]
critical_count = connection.execute("""
    SELECT count(*) FROM saved_staging
    WHERE dteday IS NULL OR hr IS NULL OR cnt IS NULL OR casual IS NULL OR registered IS NULL
       OR hr < 0 OR hr > 23
       OR cnt <> casual + registered
       OR cnt < 0 OR casual < 0 OR registered < 0
""").fetchone()[0]
bad_flags = connection.execute("""
    SELECT count(*) FROM saved_staging
    WHERE quality_weathersit_in_domain_violated IS DISTINCT FROM
       COALESCE(weathersit NOT IN (1,2,3,4) OR weathersit IS NULL
          OR season NOT IN (1,2,3,4) OR season IS NULL
          OR weekday NOT IN (0,1,2,3,4,5,6) OR weekday IS NULL, false)
""").fetchone()[0]
flagged_count = connection.execute("""
    SELECT count(*) FROM saved_staging WHERE quality_weathersit_in_domain_violated
""").fetchone()[0]
bad_reasons = connection.execute("""
    SELECT count(*) FROM saved_quarantine
    WHERE quality_rule_ids IS NULL OR quality_rule_ids = ''
       OR quarantine_reason IS NULL OR quarantine_reason = ''
""").fetchone()[0]

assert raw_count == staging_count + quarantine_count, "Не сошёлся баланс строк."
assert critical_count == 0, "В staging остались критические нарушения."
assert bad_flags == 0, "Некорректные признаки."
assert bad_reasons == 0, "В карантине есть строки без причины."

show_table(
    ["Raw", "Staging", "Карантин", "Флаги", "Критические нарушения"],
    [(raw_count, staging_count, quarantine_count, flagged_count, critical_count)],
)

Raw,Staging,Карантин,Флаги,Критические нарушения
17379,17379,0,0,0


In [ ]:
show_table(
    ["Поле", "Тип"], [row[:2] for row in connection.execute("DESCRIBE saved_quarantine").fetchall()]
)

Поле,Тип
instant,BIGINT
dteday,DATE
season,BIGINT
yr,BIGINT
mnth,BIGINT
hr,BIGINT
holiday,BIGINT
weekday,BIGINT
workingday,BIGINT
weathersit,BIGINT


In [ ]:
show_table(
    ["Поле", "Тип"], [row[:2] for row in connection.execute("DESCRIBE saved_staging").fetchall()]
)

Поле,Тип
instant,BIGINT
dteday,DATE
season,BIGINT
yr,BIGINT
mnth,BIGINT
hr,BIGINT
holiday,BIGINT
weekday,BIGINT
workingday,BIGINT
weathersit,BIGINT


После итоговой проверки контрольные количества следующие:

raw - 17379

staging - 17379

карантин - 0

флаги - 0

критические нарушения - 0